In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
dbutils.widgets.text("container", "raw")
dbutils.widgets.text("catalogo", "catalog_au")
dbutils.widgets.text("esquema", "bronze")
dbutils.widgets.text("storageName", "adlsproyect0408") 


In [0]:
container = dbutils.widgets.get("container")
catalogo = dbutils.widgets.get("catalogo")
esquema = dbutils.widgets.get("esquema")
storageName = dbutils.widgets.get("storageName")

ruta_reviews = f"abfss://{container}@{storageName}.dfs.core.windows.net/reviews.csv"


In [0]:

reviews_schema = StructType(fields=[
    StructField("Time_submitted", StringType(), True), 
    StructField("Review", StringType(), True),
    StructField("Rating", IntegerType(), True),
    StructField("Total_thumbsup", IntegerType(), True),
    StructField("Reply", StringType(), True)
])


In [0]:
df_reviews = spark.read\
    .option('header', True)\
    .option('multiLine', True) \
    .schema(reviews_schema)\
    .csv(ruta_reviews)


In [0]:
reviews_selected_df = df_reviews.select(
    col("Time_submitted"), col("Review"), col("Rating"), 
    col("Total_thumbsup"), col("Reply")
)

In [0]:
reviews_final_df = reviews_selected_df.withColumn("ingestion_date", current_timestamp())


In [0]:
reviews_final_df.write.mode("overwrite").insertInto(f"{catalogo}.{esquema}.spotify_reviews")